# Regex Checking (the `+` operator)

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Onsite Loop, Dynamic Programming, Strings · **Difficulty/Frequency:** Uncommon (3/10)

## Concepts

**What this problem is really testing:**
- Turning an ambiguous rule ("one **or more**") into a set of explicit **state transitions**
- Spotting **overlapping subproblems** — the signature of dynamic programming
- That an un-memoised branching recursion is **exponential**, not polynomial

**First-principles primer — what is each piece?**

- **The `+` operator.** `a+` means "one or more `a`s". The ambiguity is the whole difficulty: given `s = "aaa"` and `p = "a+"`, how many `a`s should the `+` swallow? You cannot know locally — it depends on what the *rest* of the pattern needs. So you try both and let the recursion sort it out.
- **The state `(i, j)`.** How far into the string, and how far into the pattern. Everything about the future depends only on these two numbers — nothing about *how* you got there matters. That property is what makes memoisation valid.
- **Overlapping subproblems.** Different sequences of choices land on the *same* `(i, j)`. Recomputing it each time is the waste; caching it is dynamic programming.

**The transitions — the whole algorithm in a table.**

At `(i, j)`, look at whether `p[j+1]` is a `+`:

| Situation | Transition |
|---|---|
| `j == len(p)` | **match iff** `i == len(s)` — *both* must finish |
| `p[j+1] == '+'` and `s[i] == p[j]` | consume one copy, then **either** `(i+1, j)` (take another) **or** `(i+1, j+2)` (done with the group) |
| `p[j+1] == '+'` and `s[i] != p[j]` | fail — `+` demands **at least one** |
| plain char, `s[i] == p[j]` | `(i+1, j+1)` |
| plain char, mismatch | fail |

**Two details that are easy to get wrong:**

- **`j + 1 < len(p)` before reading `p[j+1]`.** A pattern ending in a normal character would otherwise index off the end.
- **The base case needs *both* pointers finished.** `i == len(s)` alone would accept `"go"` against `"google"`; `j == len(p)` alone would accept `"googlexyz"` against `"google"`.

**The trap in the official answer.** It claims O(n × m) "with memoization, or the recursion tree has at most n × m distinct states without it" — and then shows code with **no memoisation**. The number of distinct *states* is n × m, but the number of *paths* through them is exponential: every `+` branches two ways, and the branches re-explore the same suffixes. `a+a+a+a+a+a+a+a+` against 30 `a`s is enough to hang it. The notebook times this directly.

**Simple worked example.** `s = "google"`, `p = "go+gle"`.

```
(0,0) 'g' vs 'g', next is 'o' not '+'  -> (1,1)
(1,1) 'o' vs 'o', next IS '+'          -> branch:
        (2,1) take another 'o'?  s[2]='o'... wait, s = g,o,o,g,l,e -> s[2]='o' YES
        (2,3) done with the group
```

For `"google"` = `g o o g l e`, the `+` must swallow **both** `o`s, so the successful path is `(1,1) -> (2,1) -> (3,3) -> ...` — take one `o`, take another, then move past the `+`. The other branch, `(2,3)`, tries to match `'g'` against `s[2]='o'` and fails. **The branch that fails first is why you cannot be greedy or lazy — you have to try both.**

## Problem Statement

Does `s` match the pattern `p`, where `+` means "the preceding character, one or more times"?

The match must cover the **entire** string.

```python
matches("google", "go+gle")   # -> True   ('o+' swallows both o's)
matches("gogle",  "go+gle")   # -> True   ('o+' swallows one)
matches("ggle",   "go+gle")   # -> False  ('+' needs at least ONE 'o')
```

### Approach 1 — Naive recursion (the official answer, and it is exponential)

**Idea:** branch on every `+`. Consume one copy, then try *both* "take another" and "move on".

Correct, and the shape you want to write first because it maps one-for-one onto the transition table. The problem is that the two branches **overlap**: they frequently reach the same `(i, j)` by different routes, and each time the whole subtree below is recomputed from scratch.

**Time complexity:** **exponential** in the number of `+` groups — *not* the O(n × m) the official answer claims. The claim confuses the number of distinct states (n × m) with the number of paths through them.

**Space complexity:** O(n + m) for the recursion stack.

In [ ]:
import sys
from functools import lru_cache
from typing import List

sys.setrecursionlimit(10000)

CALLS = {"naive": 0, "memo": 0}          # instrumentation, to make the blow-up visible


def matches_naive(s: str, p: str) -> bool:
    def dfs(i: int, j: int) -> bool:
        CALLS["naive"] += 1
        if j == len(p):
            return i == len(s)           # BOTH must be exhausted

        if j + 1 < len(p) and p[j + 1] == "+":     # bounds check BEFORE reading p[j+1]
            if i < len(s) and s[i] == p[j]:
                # take one copy, then either take another, or finish the group
                return dfs(i + 1, j) or dfs(i + 1, j + 2)
            return False                 # '+' requires at least one occurrence

        if i < len(s) and s[i] == p[j]:
            return dfs(i + 1, j + 1)
        return False

    return dfs(0, 0)

### Approach 2 — Optimal (memoise the `(i, j)` state)

**Idea:** the answer at `(i, j)` depends **only** on `i` and `j` — never on the path taken to get there. So cache it. Every state is computed once, and there are at most `(n+1) × (m+1)` of them.

That single decorator is the difference between exponential and O(n × m). It is also the textbook signature of dynamic programming: *"the future depends only on the current state"* plus *"the same states recur"*.

**Time complexity:** **O(n × m)** — each state computed once, each doing O(1) work.

**Space complexity:** O(n × m) for the cache, plus O(n + m) stack.

In [ ]:
def matches(s: str, p: str) -> bool:
    @lru_cache(maxsize=None)             # <- the ENTIRE difference from Approach 1
    def dfs(i: int, j: int) -> bool:
        CALLS["memo"] += 1
        if j == len(p):
            return i == len(s)

        if j + 1 < len(p) and p[j + 1] == "+":
            if i < len(s) and s[i] == p[j]:
                return dfs(i + 1, j) or dfs(i + 1, j + 2)
            return False

        if i < len(s) and s[i] == p[j]:
            return dfs(i + 1, j + 1)
        return False

    try:
        return dfs(0, 0)
    finally:
        dfs.cache_clear()                # the cache is per-call; do not leak it between inputs

### Approach 3 — Bottom-up DP (a table, no recursion)

**Idea:** the same recurrence, filled in as a table instead of a call stack. `dp[i][j]` = "does `s[i:]` match `p[j:]`?"

Because `dfs(i, j)` only ever calls states with a **larger** `i`, filling the table from `i = n` downwards means every dependency is already computed.

Worth having for two reasons: no recursion limit, and it makes the **rolling-array** optimisation obvious — row `i` depends only on row `i+1`, so you never need the whole table in memory. That drops space from O(n × m) to O(m), which is the answer to the last follow-up.

**Time complexity:** O(n × m).

**Space complexity:** O(m) with the rolling array (O(n × m) for the full table).

In [ ]:
def matches_dp(s: str, p: str) -> bool:
    n, m = len(s), len(p)
    # nxt[j] = does s[i+1:] match p[j:]?   cur[j] = does s[i:] match p[j:]?
    nxt = [False] * (m + 1)
    nxt[m] = True                        # empty string vs empty pattern

    for i in range(n, -1, -1):
        cur = [False] * (m + 1)
        cur[m] = (i == n)                # pattern exhausted: match only if string is too
        for j in range(m - 1, -1, -1):
            if j + 1 < m and p[j + 1] == "+":
                if i < n and s[i] == p[j]:
                    #        take another          |        finish the group
                    cur[j] = nxt[j] or nxt[j + 2] if j + 2 <= m else nxt[j]
                else:
                    cur[j] = False
            else:
                cur[j] = i < n and s[i] == p[j] and nxt[j + 1]
        nxt = cur                        # roll: only ONE row is ever kept
    return nxt[0]

### Follow-up — adding `*` (zero or more) and `?` (any single character)

**Idea:** each operator is just a different pair of transitions on the same state machine.

| Operator | Meaning | Transitions from `(i, j)` |
|---|---|---|
| `c+` | one or more `c` | must match once: `(i+1, j)` **or** `(i+1, j+2)` |
| `c*` | **zero** or more `c` | **skip entirely**: `(i, j+2)`; or if it matches, `(i+1, j)` |
| `?` | exactly one of **any** character | `(i+1, j+1)`, no character comparison |

The one structural difference is `*`: because it can match **zero** characters, it has a transition that does **not** consume any of `s`. That means `j` can advance while `i` stands still — so the recursion is no longer strictly decreasing in `i`, and you must be sure `j` strictly increases on that branch or you loop forever.

Note also that `a+` is exactly `a` followed by `a*`, which is a nice thing to point out.

**Time complexity:** O(n × m), memoised.

**Space complexity:** O(n × m).

In [ ]:
def matches_full(s: str, p: str) -> bool:
    """Supports c+ (one or more), c* (zero or more), and ? (any single char)."""
    @lru_cache(maxsize=None)
    def dfs(i: int, j: int) -> bool:
        if j == len(p):
            return i == len(s)

        # Does the current pattern char match s[i]?  '?' matches anything.
        here = i < len(s) and (p[j] == "?" or s[i] == p[j])

        if j + 1 < len(p) and p[j + 1] == "*":
            # zero copies: skip the whole group WITHOUT consuming from s
            if dfs(i, j + 2):
                return True
            return here and dfs(i + 1, j)          # or one more copy
        if j + 1 < len(p) and p[j + 1] == "+":
            return here and (dfs(i + 1, j) or dfs(i + 1, j + 2))   # at least one
        return here and dfs(i + 1, j + 1)

    try:
        return dfs(0, 0)
    finally:
        dfs.cache_clear()

## Verification

The example from the statement, then the boundary cases: `+` needing at least one, `+` at the end of the pattern, empty inputs, and a cross-check against Python's own `re` module on randomised inputs.

In [ ]:
import random
import re
import time

IMPLS = [matches, matches_naive, matches_dp]

# --- The example from the problem statement ---
for fn in IMPLS:
    assert fn("google", "go+gle") is True, fn.__name__

# --- How many copies the '+' must swallow ---
for fn in IMPLS:
    assert fn("gogle", "go+gle") is True, f"{fn.__name__}: one 'o'"
    assert fn("google", "go+gle") is True, f"{fn.__name__}: two 'o's"
    assert fn("gooogle", "go+gle") is True, f"{fn.__name__}: three 'o's"
    assert fn("ggle", "go+gle") is False, f"{fn.__name__}: '+' needs at least ONE"

# --- Basic matching, no operators ---
for fn in IMPLS:
    assert fn("abc", "abc") is True, fn.__name__
    assert fn("abc", "abd") is False, fn.__name__
    assert fn("", "") is True, fn.__name__
    assert fn("a", "") is False, f"{fn.__name__}: pattern exhausted, string is not"
    assert fn("", "a") is False, f"{fn.__name__}: string exhausted, pattern is not"
    assert fn("abc", "ab") is False, f"{fn.__name__}: the match must cover the WHOLE string"
    assert fn("ab", "abc") is False, fn.__name__

# --- '+' on its own, and at the end of the pattern ---
for fn in IMPLS:
    assert fn("", "a+") is False, f"{fn.__name__}: '+' cannot match zero characters"
    assert fn("a", "a+") is True, fn.__name__
    assert fn("aaaa", "a+") is True, fn.__name__
    assert fn("aaab", "a+") is False, fn.__name__
    assert fn("ab", "ab+") is True, f"{fn.__name__}: a '+' at the very end of the pattern"
    assert fn("abbb", "ab+") is True, fn.__name__
    assert fn("a", "ab+") is False, fn.__name__

# --- Several '+' groups, including adjacent ones ---
for fn in IMPLS:
    assert fn("aabb", "a+b+") is True, fn.__name__
    assert fn("ab", "a+b+") is True, fn.__name__
    assert fn("aab", "a+b+") is True, fn.__name__
    assert fn("b", "a+b+") is False, fn.__name__
    assert fn("a", "a+b+") is False, fn.__name__
    assert fn("aaa", "a+a+") is True, f"{fn.__name__}: adjacent groups split the run"
    assert fn("a", "a+a+") is False, f"{fn.__name__}: two groups need at least two chars"
    assert fn("aa", "a+a+") is True, fn.__name__

# --- The greedy trap: a naive "take as many as possible" would fail these ---
for fn in IMPLS:
    assert fn("aaa", "a+aa") is True, f"{fn.__name__}: '+' must leave 2 a's behind"
    assert fn("aab", "a+ab") is True, fn.__name__
    assert fn("aaab", "a+aab") is True, fn.__name__

# --- Cross-check against Python's own regex engine on randomised input ---
random.seed(59)
alphabet = "ab"
for _ in range(2500):
    s = "".join(random.choices(alphabet, k=random.randint(0, 8)))
    # Build a pattern of plain chars and c+ groups
    p_parts = []
    for _ in range(random.randint(0, 4)):
        c = random.choice(alphabet)
        p_parts.append(c + "+" if random.random() < 0.5 else c)
    p = "".join(p_parts)
    expected = re.fullmatch(p, s) is not None
    for fn in IMPLS:
        assert fn(s, p) is expected, (fn.__name__, repr(s), repr(p), expected)

# --- Follow-up: '*' and '?' ---
assert matches_full("google", "go+gle") is True, "the '+' behaviour is unchanged"
assert matches_full("ggle", "go*gle") is True, "'*' matches ZERO occurrences"
assert matches_full("ggle", "go+gle") is False, "'+' does not"
assert matches_full("google", "go*gle") is True
assert matches_full("", "a*") is True
assert matches_full("aaa", "a*") is True
assert matches_full("gxgle", "g?gle") is True, "'?' matches any single character"
assert matches_full("ggle", "g?gle") is False, "'?' needs exactly one"
assert matches_full("gxygle", "g?gle") is False
assert matches_full("abc", "a*b*c*") is True
assert matches_full("", "a*b*c*") is True

for _ in range(2000):
    s = "".join(random.choices(alphabet, k=random.randint(0, 7)))
    p_parts = []
    for _ in range(random.randint(0, 4)):
        c = random.choice(alphabet)
        r = random.random()
        p_parts.append(c + "+" if r < 0.34 else c + "*" if r < 0.67 else c)
    p = "".join(p_parts)
    expected = re.fullmatch(p, s) is not None
    assert matches_full(s, p) is expected, (repr(s), repr(p), expected)

# --- THE point: memoisation turns exponential into polynomial ---
CALLS["naive"] = CALLS["memo"] = 0
hard_s = "a" * 22
hard_p = "a+" * 8 + "b"          # every '+' branches; nothing matches, so nothing prunes

t0 = time.monotonic()
assert matches_naive(hard_s, hard_p) is False
naive_time = time.monotonic() - t0
naive_calls = CALLS["naive"]

t0 = time.monotonic()
assert matches(hard_s, hard_p) is False
memo_time = time.monotonic() - t0
memo_calls = CALLS["memo"]

states = (len(hard_s) + 1) * (len(hard_p) + 1)
assert memo_calls <= states, f"memoised calls ({memo_calls}) must not exceed states ({states})"
assert naive_calls > memo_calls * 20, (
    f"the un-memoised version should blow up: {naive_calls} vs {memo_calls} calls"
)
print(f"  un-memoised: {naive_calls:>8,} calls, {naive_time*1000:7.1f} ms")
print(f"  memoised:   {memo_calls:>8,} calls, {memo_time*1000:7.1f} ms  "
      f"(state space is {states})")

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Bottom-up DP with a rolling array.** `matches_dp` above already does this: `dp[i][j]` = "does `s[i:]` match `p[j:]`?", filled from `i = n` downwards because every transition increases `i`. Since row `i` depends only on row `i+1`, one row is enough — **O(m) space** instead of O(n × m), which is the answer to the space follow-up. It also removes the recursion limit entirely.
- **Grouped `+` like `(ab)+c`.** This is a genuine step up, because the pattern is no longer a flat character sequence — you have to **parse** it into a tree first (a literal, a group, a repetition), then match against that tree. The state stops being a simple `(i, j)` pair of integers and becomes `(string position, node in the parse tree, repetition count)`. That is the point at which hand-rolled matching stops scaling and you build a real engine — which is the next bullet.
- **How real regex engines do it.** Two families, and the difference matters. **Backtracking** engines (Perl, Java, Python's `re`) do exactly what Approach 1 does, and are exponential on adversarial patterns — this is the source of real-world **ReDoS** denial-of-service vulnerabilities. **Automaton** engines (RE2, Go's `regexp`) compile the pattern to an NFA and simulate all states at once, guaranteeing O(n × m) but giving up backreferences. Being able to name that trade-off is a strong signal.
- **Why memoisation is *valid* here.** Worth stating as a property, not a trick: the answer at `(i, j)` depends only on `i` and `j`, never on the route taken. That is the **optimal substructure / memorylessness** condition, and it is what licenses caching. Whenever a recursion's arguments fully determine its result and the same arguments recur, memoisation applies — and if the state also had to include "how many copies have I taken so far", it would not.
- **Testing against the reference implementation.** The randomised checks above compare every implementation with `re.fullmatch` on thousands of inputs. When the semantics you are asked for happen to be a **subset** of something in the standard library, that library is free, exhaustive test coverage — and it catches the cases you would not have thought to write, like `"aaa"` vs `"a+aa"`, where a greedy `+` fails but the correct answer is `True`.

## Empirical complexity check

Compare the un-memoised recursion with the memoised one on a pattern built to make every `+` branch: `a+a+a+...` against a run of `a`s ending in a character that never matches, so **nothing prunes early**.

Note the sizes below grow by only **1.33x** each step (9 → 12 → 15 → 18), not 2x — the un-memoised version blows up too fast for anything larger.

| Growth per 1.33x step | What it means |
|---|---|
| ~6x | **exponential** — the branch tree explodes far faster than the input grows |
| ~1x | polynomial with a tiny constant — O(n × m) states, each computed once |

The instrumented counters printed by the verification cell above make the same point exactly: **1.2 million** calls un-memoised versus **171** memoised, against a state space of only 414.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark


def make_hard_case(n):
    """n 'a's against n/3 '+' groups, ending in an unmatchable 'b' so nothing prunes."""
    return ("a" * n, "a+" * max(1, n // 3) + "b")


def run_naive(s, p):
    matches_naive(s, p)


def run_memo(s, p):
    matches(s, p)


def run_dp(s, p):
    matches_dp(s, p)


benchmark(
    {"Approach 1 - no memoisation (exponential)": run_naive,
     "Approach 2 - memoised O(n*m)": run_memo,
     "Approach 3 - bottom-up DP O(n*m), O(m) space": run_dp},
    make_hard_case,
    sizes=[9, 12, 15, 18],
    repeats=1,
)

## Patterns learned

- **"One or more" means branch, not loop.** You cannot know locally how many copies to take — it depends on what the rest of the pattern needs. Try both and let the search decide.
- **Write the transition table before the code.** Five rows, and the recursion falls out of it. It also makes the two off-by-ones (`j+1 < len(p)`, and "both pointers must finish") impossible to miss.
- **Overlapping subproblems + a state that determines the future = dynamic programming.** Once `(i, j)` fully determines the answer, memoisation is *valid*, and it is the difference between exponential and O(n × m).
- **A branching recursion is exponential until you prove otherwise.** Counting *states* is not counting *paths*. The official answer's complexity claim is wrong for exactly this reason, and the benchmark shows it.
- **Top-down memoisation and bottom-up DP are the same recurrence.** The table version removes the recursion limit and makes the **rolling-array** space saving obvious, because each row depends only on the next one.
- **Test against the standard library when your semantics are a subset of it.** `re.fullmatch` gave thousands of free test cases here — including `"aaa"` vs `"a+aa"`, which breaks any greedy implementation.